# DATA 221: Final Project Code

## Part A: Data Wrangling

For this project, I built a country–year panel dataset covering roughly 50 countries from 2008 to 2020. **I use annual mean PM2.5 (EN.ATM.PM25.MC.M3) as the air quality measure.**

All other variables come from the World Bank Development Indicators to keep everything at the country level and consistent across time. I selected 14 variables that represent environmental, economic, and social aspects of sustainability while also checking that data coverage was strong across countries. Below is the code that I ran to get my data: 


``` python
import requests
import pandas as pd
import time

START_YEAR = 2008
END_YEAR = 2020

COUNTRIES = [
    "USA","CAN","GBR","DEU","FRA","ITA","ESP","JPN","KOR","IND",
    "BRA","MEX","AUS","NLD","SWE","NOR","ZAF","ARG","TUR","IDN",
    "CHE","POL","BEL","AUT","DNK","FIN","IRL","PRT","GRC","NZL",
    "CHN","THA","MYS","PHL","VNM","EGY","SAU","ARE","ISR","COL",
    "PER","CHL","CZE","HUN","ROU","UKR","PAK","BGD","NGA","KEN"
]

# See Legend description in the next cell
INDICATORS = {
    "EN.ATM.PM25.MC.M3": "pm25",
    "EG.EGY.PRIM.PP.KD": "primary_energy_per_capita",
    "EG.FEC.RNEW.ZS": "renewable_energy_pct",
    "AG.LND.FRST.ZS": "forest_area_pct",
    "EG.USE.PCAP.KG.OE": "energy_use_per_capita",
    "AG.LND.AGRI.ZS": "agricultural_land_pct",
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "NV.IND.TOTL.ZS": "industry_pct_gdp",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "NE.GDI.TOTL.ZS": "gross_capital_formation_pct",
    "SP.URB.TOTL.IN.ZS": "urban_population_pct",
    "SP.DYN.LE00.IN": "life_expectancy",
    "EN.POP.DNST": "population_density",
    "SH.H2O.BASW.ZS": "basic_drinking_water_pct",
}

BASE = "https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"

def fetch_indicator(indicator_code, countries_iso3):
    countries_str = ";".join(countries_iso3)
    params = {
        "format": "json",
        "per_page": 20000,
        "date": f"{START_YEAR}:{END_YEAR}"
    }

    url = BASE.format(countries=countries_str, indicator=indicator_code)
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    payload = r.json()

    if not isinstance(payload, list) or payload[1] is None:
        return pd.DataFrame(columns=["country", "year", "value"])

    rows = []
    for item in payload[1]:
        rows.append({
            "country": item["countryiso3code"],
            "year": int(item["date"]),
            "value": item["value"]
        })

    return pd.DataFrame(rows)

dfs = []

for code, name in INDICATORS.items():
    print(f"Fetching {code}")
    dfi = fetch_indicator(code, COUNTRIES)
    dfi = dfi.rename(columns={"value": name})
    dfs.append(dfi)
    time.sleep(0.5)

df = dfs[0]
for dfi in dfs[1:]:
    df = df.merge(dfi, on=["country","year"], how="outer")

df = df.sort_values(["country","year"]).reset_index(drop=True)

print(df.head())
print("\nMissing values:")
print(df.isna().sum())

df.to_csv("clean_8_variable_panel.csv", index=False)
```

## Variable Legend 

**pm25 (EN.ATM.PM25.MC.M3)**
Annual average PM2.5 concentration. This is the main air quality outcome and serves as the pollution measure throughout the project.


### Environmental

**primary_energy_per_capita (EG.EGY.PRIM.PP.KD)**: Captures overall energy intensity and resource use.

**renewable_energy_pct (EG.FEC.RNEW.ZS)**: Share of energy coming from renewable sources, indicating cleaner energy transition.

**forest_area_pct (AG.LND.FRST.ZS)**: Forest coverage as a share of land area, reflecting environmental protection and natural assets.

**energy_use_per_capita (EG.USE.PCAP.KG.OE)**: Overall energy consumption per person, often linked to environmental pressure.

**agricultural_land_pct (AG.LND.AGRI.ZS)**: Land use for agriculture, which relates to resource use and environmental structure.


### Economic


**gdp_per_capita (NY.GDP.PCAP.KD)**: Overall economic development level.

**industry_pct_gdp (NV.IND.TOTL.ZS)**: Share of output coming from industry, often tied to emissions and energy use.

**trade_pct_gdp (NE.TRD.GNFS.ZS)**: Economic openness and integration into global markets.

**gross_capital_formation_pct (NE.GDI.TOTL.ZS)**: Investment activity and long-run development capacity.


### Social / Urban

**urban_population_pct (SP.URB.TOTL.IN.ZS)**: Level of urbanization.

**life_expectancy (SP.DYN.LE00.IN)**: General measure of population health and development.

**population_density (EN.POP.DNST)**: How concentrated the population is geographically.

**basic_drinking_water_pct (SH.H2O.BASW.ZS)**: Access to essential infrastructure and living standards.

In [11]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("clean_8_variable_panel.csv")

# 1. CLEANING: Interpolate missing values by country
df = df.sort_values(["country", "year"])

numeric_cols = df.select_dtypes(include="number").columns

df[numeric_cols] = (
    df.groupby("country")[numeric_cols]
      .apply(lambda g: g.interpolate(method="linear", limit_area="inside"))
      .reset_index(level=0, drop=True)
)

df = df.dropna().reset_index(drop=True)

# 2. SELECT USI COMPONENTS
positive_indicators = [
    'renewable_energy_pct', 
    'forest_area_pct', 
    'basic_drinking_water_pct', 
    'life_expectancy', 
    'gdp_per_capita'
]
negative_indicators = ['energy_use_per_capita']

# 3. NORMALIZATION (Min-Max Scaling to a 0-100 scale)
scaler = MinMaxScaler((0, 100))
df_scaled = df.copy()

# Scale positive indicators normally
df_scaled[positive_indicators] = scaler.fit_transform(df[positive_indicators])

# Scale negative indicators and INVERT them (so 100 is best, 0 is worst)
df_scaled[negative_indicators] = scaler.fit_transform(df[negative_indicators])
df_scaled['energy_use_per_capita'] = 100 - df_scaled['energy_use_per_capita']

# 4. CALCULATE USI (Average of the normalized indicators)
usi_columns = positive_indicators + negative_indicators
df['USI'] = df_scaled[usi_columns].mean(axis=1)

print(df[['country', 'year', 'pm25', 'USI']].head(10))

  country  year       pm25        USI
0     ARE  2008  42.957562  43.150937
1     ARE  2009  42.579407  42.100807
2     ARE  2010  42.423017  43.052147
3     ARE  2011  46.489717  43.962317
4     ARE  2012  49.858570  43.939886
5     ARE  2013  43.671732  43.638292
6     ARE  2014  38.444471  42.389466
7     ARE  2015  48.347576  41.733819
8     ARE  2016  43.081971  41.518836
9     ARE  2017  44.248506  44.303457


In [12]:
df["USI"].describe()

count    633.000000
mean      55.856601
std        9.074953
min       31.928188
25%       51.109454
50%       55.893157
75%       61.186568
max       79.550737
Name: USI, dtype: float64

In [13]:
df.isna().mean()

country                        0.0
year                           0.0
pm25                           0.0
primary_energy_per_capita      0.0
renewable_energy_pct           0.0
forest_area_pct                0.0
energy_use_per_capita          0.0
agricultural_land_pct          0.0
gdp_per_capita                 0.0
industry_pct_gdp               0.0
trade_pct_gdp                  0.0
gross_capital_formation_pct    0.0
urban_population_pct           0.0
life_expectancy                0.0
population_density             0.0
basic_drinking_water_pct       0.0
USI                            0.0
dtype: float64